# IE643 - safety layer: linear probe on the deployed encoder with sensitivity-calibrated flags (CPU)

Version `probe-v1` (2026-10-24). Accelerator None (a GPU is used if present), Internet On. No change to the student: the free-text report is still written by **K7s1_scst** alone; this notebook adds a separate, train-only-on-frozen-features classifier whose job is to **not miss** findings.

Why: the deployed model's TEST sensitivity is 0.38 pooled over the five common findings and 0 for Cardiomegaly, Consolidation, Pneumothorax and Pneumonia, because a per-report F1 reward makes calling a low-precision finding a losing bet. The encoder still ranks these findings (earlier heads: AUC 0.75-0.88), so a linear read-out of the same features with a threshold chosen for sensitivity, not F1, can flag them. The trade is explicit: more false alarms, far fewer misses (the usual design for a triage aid).

Method (fixed before running):
1. Load `distill_runs/K7s1_scst/best.pt`; sanity-check by greedy-decoding 32 TEST images and comparing with the saved `pred_test.csv` (preprocessing must match: 512 -> 384 px bilinear with antialias, mean/std 0.5).
2. Frozen encoder features per study: `clsmean` = [CLS ; mean patch token] (384-d) and `grid` = `clsmean` + the 4x4 average-pooled patch grid (3,456-d).
3. Labels: CheXbert on the GT impression (same labeller as every CE metric). Training: positive = class 1, uncertain samples left out for that label (U-Ignore); evaluation: uncertain = negative (CE convention).
4. One L2 logistic regression per label on standardised features (`class_weight="balanced"`), C in {0.01, 0.1, 1} chosen per label by VAL AUC; feature set chosen once by mean VAL AUC over labels with >= 10 VAL positives.
5. Threshold per label on VAL for a target sensitivity (0.80 / 0.90 / 0.95; **the app uses 0.90**): the largest threshold whose VAL sensitivity is >= the target.
6. **App flag set (pre-registered):** the 12 pathology labels (No Finding and Support Devices excluded), each shown only if VAL AUC >= 0.70 and VAL positives >= 10; the others are listed as "not assessed".
7. TEST, reported: per label AUC (bootstrap CI), sensitivity (Wilson CI), specificity, PPV, flag rate; flags per study; share of GT-normal studies with at least one flag (alarm burden); pooled sensitivity of **text OR flag** vs text alone.

Limitation to state: VAL has few positives for rare labels (e.g. Pneumonia about 15), so a 90% VAL target can land well below 90% on TEST; the Wilson interval shows it.

**Inputs:** the `klscst2` commit output (`distill_runs/K7s1_scst/best.pt`, `pred_val.csv`, `pred_test.csv`), IE643 Cleaning Step 7 (`reports_clean.csv`), IE643 Cleaning (`images_512.npy`). Runtime on Kaggle CPU: about 10-15 min of feature extraction (9,996 images at 384 px), ~5 min CheXbert, a few minutes of fitting.

In [ ]:
import time
T0 = time.time()
NOTEBOOK_VERSION = "probe-v2"
INPUT_ROOT, WORK = "/kaggle/input", "/kaggle/working"
DEPLOYED = "K10s1_fb2"
TARGETS = (0.80, 0.90, 0.95)
APP_TARGET = 0.90
C_GRID = (0.01, 0.1, 1.0)
FEATURE_SETS = ("clsmean", "grid")
MIN_VAL_POS, MIN_AUC = 10, 0.70          # app flag inclusion (pre-registered)
BATCH = 32
N_SANITY = 32
OFFLINE_TEST = False                     # only for testing outside Kaggle (fake tokenizer / labeller, random weights)
print("notebook version:", NOTEBOOK_VERSION)

In [ ]:
import os, sys, re, json, glob, math, dataclasses
import numpy as np, pandas as pd, torch, torch.nn.functional as F

def find_one(pat, check=None, what=""):
    c = [p for p in sorted(glob.glob(f"{INPUT_ROOT}/**/{pat}", recursive=True)) if check is None or check(p)]
    assert c, f"not found: {pat} ({what})"
    if len(c) > 1:
        print(f"  several {pat}; using {c[0]}")
    return c[0]
CSV = find_one("reports_clean.csv", lambda p: "impression_final" in pd.read_csv(p, nrows=0).columns, "attach IE643 Cleaning Step 7")
NPY = find_one("images_512.npy", what="attach IE643 Cleaning")
CKPT = find_one(f"distill_runs/{DEPLOYED}/best.pt", what="attach the klscst2 commit output")
RUN_DIR = os.path.dirname(CKPT)
df = pd.read_csv(CSV, low_memory=False)
df["impression_final"] = df["impression_final"].fillna("").astype(str)
df["use_for_training"] = df["use_for_training"].astype(str).str.lower().isin(["true", "1"])
U = df[df.use_for_training]
SPL = {s: U[U.subset == s].reset_index(drop=True) for s in ("train", "val", "test")}
imgs = np.load(NPY, mmap_mode="r")
assert len(df) == imgs.shape[0]
PRED = {s: pd.read_csv(f"{RUN_DIR}/pred_{s}.csv", keep_default_na=False).drop_duplicates("idx").set_index("idx") for s in ("val", "test")}
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"csv {CSV}\nnpy {NPY}\nckpt {CKPT}\nsplits", {s: len(d) for s, d in SPL.items()}, "| device", dev)

In [ ]:
# ---- the student (model.py embedded verbatim from distill-v9.1) and its deployed weights
MODEL_PY = r'''"""IE643 student: ViT-Tiny encoder -> connector -> DistilGPT2 decoder (+ distillation / clinical heads).

Two ways to condition the decoder on the image (cfg.mode):
  "xattn"  : cross-attention inside every DistilGPT2 block (CvT2DistilGPT2 style). Main model.
  "prefix" : a Perceiver-style resampler turns the patch tokens into `n_prefix` vectors that are
             prepended to the text embeddings (ClipCap / LLaVA style). Ablation. With
             set_stage("align") kept for the whole run this is the frozen-LM prefix-tuning ablation.

Conventions
  * BOS == EOS == 50256 (GPT-2 has no BOS). Student vocab = 50257, teacher ids >= 50257 are dropped in the loss.
  * forward() returns logits ALIGNED with labels: logits[:, j] predicts labels[:, j]
    (input = [BOS, t0..t_{n-1}], labels = [t0..t_{n-1}, EOS]). No internal shift anywhere.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2Config, GPT2LMHeadModel

EOS_ID = 50256
VOCAB = 50257


@dataclass
class StudentConfig:
    # encoder
    # Main setting: native-384 checkpoint (576 patch tokens, 24x24 grid -> clean 4x4 pooling).
    # 224 ablation: encoder_name="vit_tiny_patch16_224.augreg_in21k_ft_in1k", img_size=224.
    # Keep encoder_name/img_size identical to the NN-retrieval baseline you compare against.
    encoder_name: str = "vit_tiny_patch16_384.augreg_in21k_ft_in1k"
    img_size: int = 384
    pretrained_enc: bool = True
    # decoder
    decoder_name: str = "distilgpt2"
    pretrained_dec: bool = True
    mode: str = "xattn"            # "xattn" | "prefix"
    n_prefix: int = 32             # prefix mode: number of visual prefix tokens
    resampler_heads: int = 8
    # distillation targets (cached teacher features)
    d_grid: int = 1024             # siglip_grid4  [16, 1024]
    d_mean: int = 1024             # siglip_mean   [1024]
    d_proj: int = 2560             # proj_mean     [2560]
    grid_hw: int = 4
    # clinical head
    n_clin: int = 14
    diag_prompt: bool = False      # A10: head predictions -> 14 prompt tokens between the visual prefix and BOS (prefix mode only)
    drop: float = 0.1
    max_len: int = 192             # max text tokens incl. BOS. EDA: median impression 86 tok, p95 187 (pre-cleaning);
                                   # set from the p95 of ntok_final in reports_clean.csv before training.


class Connector(nn.Module):
    """Linear(D_v->H) -> GELU -> Linear(H->H) -> LayerNorm (LLaVA-1.5 style 2-layer MLP)."""

    def __init__(self, d_in: int, d_out: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d_out), nn.GELU(), nn.Linear(d_out, d_out))
        self.norm = nn.LayerNorm(d_out)

    def forward(self, x):
        return self.norm(self.net(x))


class Resampler(nn.Module):
    """m learned queries cross-attend to the visual tokens -> [B, m, H] (Perceiver / Q-Former lite)."""

    def __init__(self, d: int, n_queries: int, n_heads: int, drop: float = 0.0):
        super().__init__()
        self.queries = nn.Parameter(torch.randn(n_queries, d) * 0.02)
        self.ln_q, self.ln_kv, self.ln_ff = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=drop, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, kv):
        q = self.queries.unsqueeze(0).expand(kv.size(0), -1, -1)
        kv = self.ln_kv(kv)
        x = q + self.attn(self.ln_q(q), kv, kv, need_weights=False)[0]
        return x + self.ff(self.ln_ff(x))


class XRayReportStudent(nn.Module):
    def __init__(self, cfg: StudentConfig):
        super().__init__()
        assert cfg.mode in ("xattn", "prefix")
        self.cfg = cfg

        # ---- encoder -------------------------------------------------------------------------
        self.encoder = timm.create_model(
            cfg.encoder_name, pretrained=cfg.pretrained_enc, num_classes=0, img_size=cfg.img_size
        )
        self.n_prefix_tok = getattr(self.encoder, "num_prefix_tokens", 1)  # CLS (+ reg tokens)
        d_v = self.encoder.num_features
        self.d_v = d_v
        self.enc_cfg = self.encoder.pretrained_cfg  # mean/std for preprocessing

        # ---- decoder -------------------------------------------------------------------------
        xattn = cfg.mode == "xattn"
        if cfg.pretrained_dec:
            self.decoder = GPT2LMHeadModel.from_pretrained(cfg.decoder_name, add_cross_attention=xattn)
        else:  # offline / unit-test path: distilgpt2 geometry, random init
            gcfg = GPT2Config(n_layer=6, n_head=12, n_embd=768, vocab_size=VOCAB, n_positions=1024,
                              add_cross_attention=xattn)
            self.decoder = GPT2LMHeadModel(gcfg)
        self.d_t = self.decoder.config.n_embd

        # ---- connector -----------------------------------------------------------------------
        self.connector = Connector(d_v, self.d_t)
        self.resampler = (
            Resampler(self.d_t, cfg.n_prefix, cfg.resampler_heads, cfg.drop) if cfg.mode == "prefix" else None
        )

        # ---- auxiliary heads (train only; dropped at export) --------------------------------
        self.grid_head = nn.Sequential(nn.LayerNorm(d_v), nn.Linear(d_v, cfg.d_grid))
        self.mean_head = nn.Sequential(nn.LayerNorm(d_v), nn.Linear(d_v, cfg.d_mean))
        self.proj_head = nn.Sequential(nn.LayerNorm(d_v), nn.Linear(d_v, cfg.d_proj))
        self.clin_head = nn.Sequential(
            nn.LayerNorm(2 * d_v), nn.Dropout(cfg.drop), nn.Linear(2 * d_v, d_v), nn.GELU(),
            nn.Linear(d_v, cfg.n_clin),
        )

        # ---- diagnosis prompts (A10): token_c = LayerNorm(label_c + state[s_c]), s_c in {0 = not positive, 1 = positive} ---
        if cfg.diag_prompt:
            assert cfg.mode == "prefix", "diagnosis prompts are implemented for the prefix connector only"
            self.prompt_label = nn.Parameter(torch.randn(cfg.n_clin, self.d_t) * 0.02)
            self.prompt_state = nn.Parameter(torch.randn(2, self.d_t) * 0.02)
            self.prompt_ln = nn.LayerNorm(self.d_t)
            # per-label decision thresholds of the head, set on VAL by the worker; a buffer so they are saved with the weights
            self.register_buffer("prompt_thr", torch.full((cfg.n_clin,), 0.5))

    # ------------------------------------------------------------------------------------------
    def encode(self, images):
        """images [B,3,H,W] (already normalised) -> dict of visual tokens and head predictions."""
        feats = self.encoder.forward_features(images)              # [B, npt+N, D]
        cls = feats[:, 0]
        patches = feats[:, self.n_prefix_tok:]                     # [B, N, D]
        B, N, D = patches.shape
        g = int(math.isqrt(N))
        assert g * g == N, "patch tokens must form a square grid"
        mean = patches.mean(1)
        grid = F.adaptive_avg_pool2d(patches.transpose(1, 2).reshape(B, D, g, g), self.cfg.grid_hw)
        grid = grid.flatten(2).transpose(1, 2)                     # [B, 16, D] row-major
        memory = self.connector(feats)                             # [B, npt+N, H]
        if self.resampler is not None:
            memory = self.resampler(memory)                        # [B, m, H]
        return {"memory": memory, "cls": cls, "mean": mean, "grid": grid}

    def heads(self, enc):
        return {
            "grid_pred": self.grid_head(enc["grid"]),              # [B,16,d_grid]
            "mean_pred": self.mean_head(enc["mean"]),              # [B,d_mean]
            "proj_pred": self.proj_head(enc["mean"]),              # [B,d_proj]
            "clin_logits": self.clin_head(torch.cat([enc["cls"], enc["mean"]], -1)),  # [B,n_clin]
        }

    def clin_probs(self, enc):
        return torch.sigmoid(self.clin_head(torch.cat([enc["cls"], enc["mean"]], -1)).float())   # [B, n_clin] fp32

    def prompt_embeds(self, states):
        """states [B, n_clin] in {0,1} -> prompt tokens [B, n_clin, H]."""
        s = states.long().clamp(0, 1)
        return self.prompt_ln(self.prompt_label.unsqueeze(0) + self.prompt_state[s])

    def head_states(self, enc):
        return (self.clin_probs(enc).detach() >= self.prompt_thr).long()

    def forward(self, images, input_ids, attention_mask=None, with_heads=True, prompt_states=None, prompt_gt=None, p_gt=0.0):
        """prompt_states: explicit [B, n_clin] states (diag_prompt only); otherwise the head's own thresholded predictions,
        replaced per study by prompt_gt with probability p_gt (training)."""
        enc = self.encode(images)
        mem = enc["memory"]
        hd = self.heads(enc) if (with_heads or self.cfg.diag_prompt) else {}
        if self.cfg.diag_prompt:
            if prompt_states is None:
                prompt_states = (torch.sigmoid(hd["clin_logits"].detach().float()) >= self.prompt_thr).long()
                if prompt_gt is not None and p_gt > 0:
                    use_gt = torch.rand(prompt_states.size(0), 1, device=prompt_states.device) < p_gt
                    prompt_states = torch.where(use_gt, (prompt_gt > 0.5).long(), prompt_states)
            mem = torch.cat([mem, self.prompt_embeds(prompt_states).to(mem.dtype)], 1)
        if self.cfg.mode == "xattn":
            out = self.decoder(input_ids=input_ids, attention_mask=attention_mask,
                               encoder_hidden_states=mem, use_cache=False)
            logits = out.logits
        else:
            m = mem.size(1)
            emb = torch.cat([mem, self.decoder.transformer.wte(input_ids)], 1)
            am = None
            if attention_mask is not None:
                am = torch.cat([attention_mask.new_ones(attention_mask.size(0), m), attention_mask], 1)
            out = self.decoder(inputs_embeds=emb, attention_mask=am, use_cache=False)
            logits = out.logits[:, m:]                              # drop prefix positions
        res = {"logits": logits}
        if with_heads:
            res.update(hd)
        return res

    # ------------------------------------------------------------------------------------------
    @torch.no_grad()
    def generate(self, images, max_new_tokens=60, num_beams=3, no_repeat_ngram_size=3,
                 length_penalty=1.0, **kw):
        """Beam (or greedy if num_beams=1) decoding with KV cache. Returns list[list[int]] without BOS/EOS."""
        assert not self.cfg.diag_prompt, "use greedy() for diagnosis-prompt models"
        was_training = self.training
        self.eval()
        enc = self.encode(images)
        mem = enc["memory"]
        B = mem.size(0)
        bos = torch.full((B, 1), EOS_ID, dtype=torch.long, device=mem.device)
        common = dict(max_new_tokens=max_new_tokens, num_beams=num_beams, do_sample=False,
                      no_repeat_ngram_size=no_repeat_ngram_size, length_penalty=length_penalty,
                      eos_token_id=EOS_ID, pad_token_id=EOS_ID, early_stopping=True, **kw)
        if self.cfg.mode == "xattn":
            ids = self.decoder.generate(input_ids=bos, encoder_hidden_states=mem, **common)
            seqs = ids[:, 1:]                                       # strip BOS
        else:
            emb = torch.cat([mem, self.decoder.transformer.wte(bos)], 1)
            seqs = self.decoder.generate(inputs_embeds=emb, **common)  # returns only new tokens
        out = []
        for s in seqs.tolist():
            if EOS_ID in s:
                s = s[: s.index(EOS_ID)]
            out.append(s)
        self.train(was_training)
        return out

    @torch.no_grad()
    def greedy(self, images, max_new_tokens=128):
        """Greedy decoding with the decoder's KV cache, written by hand so it does not depend on the
        transformers version's generate() handling of encoder_hidden_states / inputs_embeds.
        Returns list[list[int]] without BOS/EOS."""
        was_training = self.training
        self.eval()
        mem = self.encode(images)["memory"]
        B = mem.size(0)
        cur = torch.full((B, 1), EOS_ID, dtype=torch.long, device=mem.device)
        if self.cfg.mode == "xattn":
            out = self.decoder(input_ids=cur, encoder_hidden_states=mem, use_cache=True)
        else:
            out = self.decoder(inputs_embeds=torch.cat([mem, self.decoder.transformer.wte(cur)], 1), use_cache=True)
        seqs = []
        done = torch.zeros(B, dtype=torch.bool, device=mem.device)
        for _ in range(max_new_tokens):
            nxt = out.logits[:, -1, :VOCAB].argmax(-1)
            nxt = torch.where(done, torch.full_like(nxt, EOS_ID), nxt)
            seqs.append(nxt)
            done |= nxt == EOS_ID
            if bool(done.all()):
                break
            kw = dict(input_ids=nxt[:, None], past_key_values=out.past_key_values, use_cache=True)
            if self.cfg.mode == "xattn":
                kw["encoder_hidden_states"] = mem
            out = self.decoder(**kw)
        res = []
        for s in torch.stack(seqs, 1).tolist():
            res.append(s[: s.index(EOS_ID)] if EOS_ID in s else s)
        self.train(was_training)
        return res

    @torch.no_grad()
    def greedy(self, images, max_new_tokens=128, prompt_states=None):
        """Greedy decoding with an explicit KV-cache loop (does not rely on HF generate(), whose handling of
        encoder_hidden_states for decoder-only GPT-2 differs across transformers versions).
        Returns (list[list[int]] without BOS/EOS, list[bool] hit_cap = no EOS within max_new_tokens)."""
        was_training = self.training
        self.eval()
        enc = self.encode(images)
        mem = enc["memory"]
        if self.cfg.diag_prompt:
            st = prompt_states if prompt_states is not None else self.head_states(enc)
            mem = torch.cat([mem, self.prompt_embeds(st).to(mem.dtype)], 1)
        B, dev = mem.size(0), mem.device
        xattn = self.cfg.mode == "xattn"
        bos = torch.full((B, 1), EOS_ID, dtype=torch.long, device=dev)
        if xattn:
            out = self.decoder(input_ids=bos, encoder_hidden_states=mem, use_cache=True)
        else:
            out = self.decoder(inputs_embeds=torch.cat([mem, self.decoder.transformer.wte(bos)], 1), use_cache=True)
        past, nxt = out.past_key_values, out.logits[:, -1].argmax(-1)
        done = torch.zeros(B, dtype=torch.bool, device=dev)
        steps = []
        for step in range(max_new_tokens):
            nxt = torch.where(done, torch.full_like(nxt, EOS_ID), nxt)
            steps.append(nxt)
            done = done | (nxt == EOS_ID)
            if bool(done.all()) or step == max_new_tokens - 1:
                break
            kw = dict(input_ids=nxt[:, None], past_key_values=past, use_cache=True)
            if xattn:
                kw["encoder_hidden_states"] = mem
            out = self.decoder(**kw)
            past, nxt = out.past_key_values, out.logits[:, -1].argmax(-1)
        toks = torch.stack(steps, 1).tolist()
        seqs, capped = [], []
        for s in toks:
            if EOS_ID in s:
                seqs.append(s[: s.index(EOS_ID)]); capped.append(False)
            else:
                seqs.append(s); capped.append(True)
        self.train(was_training)
        return seqs, capped

    # ------------------------------------------------------------------------------------------
    def _is_pretrained(self, name: str) -> bool:
        if name.startswith("encoder."):
            return True
        return name.startswith("decoder.") and "crossattn" not in name and "ln_cross_attn" not in name

    def set_stage(self, stage: str):
        """'align': freeze pretrained encoder+decoder, train connector / cross-attn / resampler / heads.
        'full'  : train everything (use a lower LR on pretrained parts via param_groups)."""
        assert stage in ("align", "full")
        for n, p in self.named_parameters():
            p.requires_grad = True if stage == "full" else not self._is_pretrained(n)

    def param_groups(self, lr_new=1e-4, lr_pre=3e-5, wd=0.01):
        buckets = {(pre, dec): [] for pre in (True, False) for dec in (True, False)}
        for n, p in self.named_parameters():
            if p.requires_grad:
                buckets[(self._is_pretrained(n), p.ndim >= 2 and "wpe" not in n)].append(p)
        groups = []
        for (pre, dec), ps in buckets.items():
            if ps:
                groups.append({"params": ps, "lr": lr_pre if pre else lr_new, "weight_decay": wd if dec else 0.0})
        return groups

    def size_report(self):
        def cnt(m):
            return sum(p.numel() for p in m.parameters())
        wte = self.decoder.transformer.wte.weight.numel()
        r = {
            "encoder": cnt(self.encoder),
            "connector": cnt(self.connector) + (cnt(self.resampler) if self.resampler is not None else 0),
            "decoder_total": cnt(self.decoder),
            "decoder_token_embedding": wte,
            "train_only_heads": sum(cnt(m) for m in (self.grid_head, self.mean_head, self.proj_head, self.clin_head)),
        }
        r["deploy_total"] = r["encoder"] + r["connector"] + r["decoder_total"]
        if self.cfg.diag_prompt:   # the head and the prompt tokens are part of the deployed graph for A10
            r["clin_head_and_prompts_deployed"] = cnt(self.clin_head) + self.prompt_label.numel() + self.prompt_state.numel() + cnt(self.prompt_ln)
            r["deploy_total"] += r["clin_head_and_prompts_deployed"]
        r["deploy_total_after_vocab_prune_to_8k"] = r["deploy_total"] - wte + 8000 * self.d_t
        return r
'''
SRC = f"{WORK}/probe_src"; os.makedirs(SRC, exist_ok=True)
open(f"{SRC}/model.py", "w").write(MODEL_PY)
sys.path.insert(0, SRC)
from model import StudentConfig, XRayReportStudent
ck = torch.load(CKPT, map_location="cpu", weights_only=False)
fields = {f.name for f in dataclasses.fields(StudentConfig)}
scfg = StudentConfig(**{k: v for k, v in ck["student_config"].items() if k in fields})
scfg.pretrained_enc = scfg.pretrained_dec = False                 # all weights come from the checkpoint
model = XRayReportStudent(scfg)
model.load_state_dict({k: (t.float() if t.is_floating_point() else t) for k, t in ck["state_dict"].items()}, strict=True)
model.to(dev).eval()
IMG = scfg.img_size
MEAN = torch.tensor(model.enc_cfg["mean"], device=dev).view(1, 3, 1, 1); STD = torch.tensor(model.enc_cfg["std"], device=dev).view(1, 3, 1, 1)
print(f"loaded {DEPLOYED} (selected step {ck.get('best_epoch')}) | mode {scfg.mode} | encoder {scfg.encoder_name} @ {IMG} | mean/std {model.enc_cfg['mean']}/{model.enc_cfg['std']}")

def prep(u8):
    """uint8 [B,512,512] -> normalised [B,3,IMG,IMG]: identical to distill_worker.prep without augmentation."""
    x = torch.as_tensor(u8).to(dev).float().div_(255).unsqueeze(1)
    x = F.interpolate(x, size=(IMG, IMG), mode="bilinear", antialias=True, align_corners=False)
    return (x.expand(-1, 3, -1, -1) - MEAN) / STD

amp = dev.type == "cuda"
# sanity: greedy decode a few TEST images, compare with the saved predictions of this run
if OFFLINE_TEST:
    decode = lambda s: " ".join(f"w{t % 40}" for t in s)
else:
    from transformers import AutoTokenizer
    _tok = AutoTokenizer.from_pretrained("distilgpt2"); decode = lambda s: _tok.decode(s, skip_special_tokens=True).strip()
sidx = SPL["test"].idx.astype(int).values[:N_SANITY]
with torch.no_grad(), torch.autocast(dev.type, dtype=torch.float16, enabled=amp):
    seqs, _ = model.greedy(prep(imgs[np.sort(sidx)]), max_new_tokens=int(ck["run_cfg"]["gen_max"]))
gen = dict(zip(np.sort(sidx).tolist(), [decode(s) for s in seqs]))
match = np.mean([gen[i] == PRED["test"].loc[i, "pred"] for i in sidx])
print(f"sanity decode: {match:.0%} of {len(sidx)} TEST outputs identical to pred_test.csv (fp16 GPU vs this device can flip a few tokens)")
for i in sidx[:2]:
    print(f"  saved: {PRED['test'].loc[i, 'pred'][:110]!r}\n  now  : {gen[i][:110]!r}")
if not OFFLINE_TEST:
    assert match >= 0.2, "almost every decode differs from the saved predictions: weights or preprocessing do not match the training run"
    if match < 0.7:
        print("WARNING: fewer than 70% identical - check the two examples above before trusting the features")

In [ ]:
# ---- frozen encoder features for every study (train / val / test)
@torch.no_grad()
def encode_split(d):
    idx = d.idx.astype(int).values; out = {"cls": [], "mean": [], "grid": []}
    order = np.argsort(idx); t = time.time()
    for s in range(0, len(idx), BATCH):
        sel = idx[order[s:s + BATCH]]
        with torch.autocast(dev.type, dtype=torch.float16, enabled=amp):
            e = model.encode(prep(imgs[sel]))
        for k in out:
            out[k].append(e[k].float().cpu().numpy())
        if s == 0 or (s // BATCH) % 50 == 0:
            el = time.time() - t; print(f"    {s + len(sel)}/{len(idx)} | {el / 60:.1f} min | ~{el / (s + len(sel)) * (len(idx) - s - len(sel)) / 60:.1f} min left", flush=True)
    inv = np.empty_like(order); inv[order] = np.arange(len(order))
    return {k: np.concatenate(v)[inv] for k, v in out.items()}
FEAT_FILE = f"{WORK}/probe_features_{DEPLOYED}.npz"
FT = {}
for s, d in SPL.items():
    print(f"  encoding {s} ({len(d)})"); FT[s] = encode_split(d)
np.savez_compressed(FEAT_FILE, **{f"{s}_{k}": v for s in FT for k, v in FT[s].items()}, **{f"{s}_idx": SPL[s].idx.values for s in SPL})
def featset(s, name):
    f = FT[s]; base = [f["cls"], f["mean"]]
    return np.concatenate(base + ([f["grid"].reshape(len(f["grid"]), -1)] if name == "grid" else []), 1)
print("feature dims:", {n: featset("val", n).shape[1] for n in FEATURE_SETS}, f"| saved {FEAT_FILE} | {(time.time() - T0) / 60:.1f} min so far")

In [ ]:
# ---- labels: CheXbert on the GT impression (targets) and on the deployed model's generated text (text-only calls, VAL / TEST)
CHEXBERT_LABELER_PY = r'''"""CheXbert labeler (Smit et al. 2020) for scoring generated report text.

Same architecture and checkpoint as the `f1chexbert` package (StanfordAIMI/RRG_scorers/chexbert.pth) but:
  * strict weight loading (only the known `position_ids` buffer may differ between transformers versions),
  * batched inference,
  * returns the raw 4-way classes so both label conventions can be computed:
      head classes 0 blank, 1 positive, 2 negative, 3 uncertain  (No Finding head: 0 blank, 1 positive)
      "pos_only"  : positive = class 1; uncertain counts as negative  (matches the IE643 baselines)
      "rrg"       : positive = class 1 or 3 (uncertain counts as positive; f1chexbert / RRG-paper convention)
"""
import numpy as np
import torch
import torch.nn as nn

CHEXBERT_ORDER = ["Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion", "Edema",
                  "Consolidation", "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion", "Pleural Other",
                  "Fracture", "Support Devices", "No Finding"]


class _Labeler(nn.Module):
    def __init__(self, bert):
        super().__init__()
        self.bert = bert
        h = bert.config.hidden_size
        self.linear_heads = nn.ModuleList([nn.Linear(h, 4) for _ in range(13)] + [nn.Linear(h, 2)])

    def forward(self, ids, att):
        cls = self.bert(ids, attention_mask=att)[0][:, 0]
        return [hd(cls) for hd in self.linear_heads]


class CheXbert:
    def __init__(self, device="cpu", checkpoint=None, random_init=False):
        from transformers import AutoConfig, AutoModel, BertTokenizer
        self.device = torch.device(device)
        self.tok = BertTokenizer.from_pretrained("bert-base-uncased")
        self.model = _Labeler(AutoModel.from_config(AutoConfig.from_pretrained("bert-base-uncased")))
        if not random_init:
            if checkpoint is None:
                from huggingface_hub import hf_hub_download
                checkpoint = hf_hub_download("StanfordAIMI/RRG_scorers", "chexbert.pth")
            sd = torch.load(checkpoint, map_location="cpu", weights_only=False)
            sd = sd.get("model_state_dict", sd)
            sd = {k.replace("module.", "", 1): v for k, v in sd.items()}
            res = self.model.load_state_dict(sd, strict=False)
            bad_missing = [k for k in res.missing_keys if not k.endswith("position_ids")]
            bad_unexp = [k for k in res.unexpected_keys if not k.endswith("position_ids")]
            assert not bad_missing and not bad_unexp, f"CheXbert weights mismatch: missing {bad_missing[:5]} unexpected {bad_unexp[:5]}"
        self.model.to(self.device).eval()

    @staticmethod
    def _clean(t):
        return " ".join(str(t).strip().replace("\n", " ").split())

    @torch.no_grad()
    def classes(self, texts, bs=64):
        """-> int array [N,14] of raw classes in CHEXBERT_ORDER."""
        out = []
        for s in range(0, len(texts), bs):
            chunk = [self._clean(t) for t in texts[s:s + bs]]
            # [CLS] + wordpieces + [SEP], truncated to 512 (same ids as the reference tokenize -> encode_plus path;
            # encode_plus was removed in transformers 5)
            enc = [self.tok(t, add_special_tokens=True, truncation=True, max_length=512)["input_ids"] if t else
                   [self.tok.cls_token_id, self.tok.sep_token_id] for t in chunk]
            n = max(len(e) for e in enc)
            ids = torch.zeros(len(enc), n, dtype=torch.long)
            att = torch.zeros(len(enc), n, dtype=torch.float)
            for i, e in enumerate(enc):
                ids[i, :len(e)] = torch.tensor(e); att[i, :len(e)] = 1
            logits = self.model(ids.to(self.device), att.to(self.device))
            out.append(torch.stack([l.argmax(1) for l in logits], 1).cpu())
        return torch.cat(out).numpy()

    @staticmethod
    def binary(cls, labels, convention="pos_only"):
        """raw classes [N,14] -> 0/1 matrix [N,len(labels)] in the order of `labels`."""
        J = [CHEXBERT_ORDER.index(c) for c in labels]
        c = cls[:, J]
        pos = (c == 1) if convention == "pos_only" else ((c == 1) | (c == 3))
        return pos.astype(np.int8)
'''
open(f"{SRC}/chexbert_labeler.py", "w").write(CHEXBERT_LABELER_PY)
LABELS = ["No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion", "Edema", "Consolidation",
          "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices"]
FIVE = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
PATH12 = [c for c in LABELS if c not in ("No Finding", "Support Devices")]
J5 = [LABELS.index(c) for c in FIVE]; J12 = [LABELS.index(c) for c in PATH12]
if OFFLINE_TEST:
    classes = lambda ts: np.array([[(hash(t) >> k) % 4 for k in range(14)] for t in ts])
    to_bin = lambda c: (c == 1)
else:
    from chexbert_labeler import CheXbert
    _cx = CheXbert(device=str(dev)); classes = _cx.classes
    to_bin = lambda c: CheXbert.binary(c, LABELS, "pos_only").astype(bool)
    J_ORDER = None
t = time.time()
GCLS = {s: classes(d.impression_final.tolist()) for s, d in SPL.items()}
if not OFFLINE_TEST:                      # raw classes come in CheXbert's own order; re-order to LABELS
    from chexbert_labeler import CHEXBERT_ORDER
    GCLS = {s: c[:, [CHEXBERT_ORDER.index(l) for l in LABELS]] for s, c in GCLS.items()}
Y = {s: (c == 1) for s, c in GCLS.items()}                  # positive
UNC = {s: (c == 3) for s, c in GCLS.items()}                # uncertain (left out of training only)
UNC = {s: np.where(np.arange(14)[None] == LABELS.index("No Finding"), False, u) for s, u in UNC.items()}
TXT = {}
for s in ("val", "test"):
    preds = PRED[s].loc[SPL[s].idx.astype(int).values, "pred"].astype(str).tolist()
    c = classes(preds)
    if not OFFLINE_TEST:
        c = c[:, [CHEXBERT_ORDER.index(l) for l in LABELS]]
    TXT[s] = (c == 1)
print(f"CheXbert: {(time.time() - t) / 60:.1f} min | positives per label (train / val / test):")
print(pd.DataFrame({s: Y[s].sum(0) for s in Y}, index=LABELS).to_string())

In [ ]:
# ---- logistic-regression probes: C per label by VAL AUC; feature set by mean VAL AUC (labels with >= MIN_VAL_POS VAL positives)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings("ignore")
def auc(y, s):
    return float(roc_auc_score(y, s)) if 0 < y.sum() < len(y) else float("nan")
FITS = {}
for fs in FEATURE_SETS:
    Xtr, Xva = featset("train", fs), featset("val", fs)
    sc = StandardScaler().fit(Xtr); Xtr, Xva = sc.transform(Xtr), sc.transform(Xva)
    per = {}
    for j, c in enumerate(LABELS):
        keep = ~UNC["train"][:, j]; y = Y["train"][keep, j]
        if y.sum() < 5:
            per[c] = None; continue
        best = None
        for C in C_GRID:
            m = LogisticRegression(C=C, class_weight="balanced", max_iter=3000).fit(Xtr[keep], y)
            a = auc(Y["val"][:, j], m.decision_function(Xva))
            if best is None or (a == a and a > best[0]):
                best = (a, C, m)
        per[c] = {"val_auc": best[0], "C": best[1], "model": best[2]}
    ok = [c for c in LABELS if per[c] and Y["val"][:, LABELS.index(c)].sum() >= MIN_VAL_POS]
    FITS[fs] = {"scaler": sc, "per": per, "mean_val_auc": float(np.nanmean([per[c]["val_auc"] for c in ok]))}
    print(f"feature set {fs}: mean VAL AUC {FITS[fs]['mean_val_auc']:.3f} over {len(ok)} labels | " +
          ", ".join(f"{c} {per[c]['val_auc']:.2f}" for c in LABELS if per[c]))
FS = max(FITS, key=lambda k: FITS[k]["mean_val_auc"])
P = FITS[FS]
print(f"-> feature set {FS} (pre-registered choice on VAL)")
SC = {s: np.full((len(SPL[s]), 14), np.nan) for s in ("val", "test")}
for s in SC:
    X = P["scaler"].transform(featset(s, FS))
    for j, c in enumerate(LABELS):
        if P["per"][c]:
            SC[s][:, j] = P["per"][c]["model"].decision_function(X)

In [ ]:
# ---- thresholds on VAL for target sensitivity; TEST evaluation
def thr_for_sens(scores, y, target):
    pos = np.sort(scores[y])
    if len(pos) == 0:
        return float("nan")
    return float(pos[int(math.floor((1 - target) * len(pos)))])      # largest t with VAL sensitivity >= target (scores >= t are flagged)
def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)
rs = np.random.RandomState(0); BOOT = [rs.randint(0, len(SPL["test"]), len(SPL["test"])) for _ in range(500)]
THRS = {t: np.full(14, np.nan) for t in TARGETS}
rows, IN_APP = [], []
for j, c in enumerate(LABELS):
    if not P["per"][c]:
        rows.append({"label": c, "note": "too few train positives"}); continue
    yv, yt, st = Y["val"][:, j], Y["test"][:, j], SC["test"][:, j]
    for t in TARGETS:
        THRS[t][j] = thr_for_sens(SC["val"][:, j], yv, t)
    a = auc(yt, st); ab = [auc(yt[b], st[b]) for b in BOOT]
    in_app = c in PATH12 and P["per"][c]["val_auc"] >= MIN_AUC and yv.sum() >= MIN_VAL_POS
    if in_app:
        IN_APP.append(c)
    r = {"label": c, "val_pos": int(yv.sum()), "test_pos": int(yt.sum()), "val_AUC": P["per"][c]["val_auc"], "C": P["per"][c]["C"],
         "test_AUC": a, "test_AUC_CI": f"[{np.nanpercentile(ab, 2.5):.2f}, {np.nanpercentile(ab, 97.5):.2f}]", "in_app": in_app}
    for t in TARGETS:
        f = st >= THRS[t][j]; tp = int((f & yt).sum()); lo, hi = wilson(tp, int(yt.sum()))
        r[f"sens@{t:.2f}"] = f"{tp / max(yt.sum(), 1):.2f} [{lo:.2f}, {hi:.2f}]"
        r[f"spec@{t:.2f}"] = float((~f & ~yt).sum() / max((~yt).sum(), 1))
        r[f"PPV@{t:.2f}"] = float(tp / max(f.sum(), 1))
        r[f"flag_rate@{t:.2f}"] = float(f.mean())
    tx = TXT["test"][:, j]
    r["text_sens"] = float((tx & yt).sum() / max(yt.sum(), 1))
    rows.append(r)
TABLE = pd.DataFrame(rows).set_index("label")
pd.set_option("display.width", 250)
print(f"TEST, probe on {DEPLOYED} encoder ({FS} features); thresholds set on VAL for target sensitivity; text_sens = the generated report alone:")
print(TABLE[[c for c in TABLE.columns if not c.startswith(("PPV@0.80", "flag_rate@0.80", "spec@0.80"))]].to_string(float_format=lambda v: f"{v:.3f}"))
print(f"\napp flag set (12 pathologies, VAL AUC >= {MIN_AUC}, VAL positives >= {MIN_VAL_POS}): {IN_APP}")
print(f"not assessed in the app: {[c for c in PATH12 if c not in IN_APP]}")

# system-level numbers at the app target
JA = [LABELS.index(c) for c in IN_APP]
FL = {s: (SC[s][:, JA] >= THRS[APP_TARGET][JA]) for s in ("val", "test")}
yt = Y["test"]; normal = Y["test"][:, LABELS.index("No Finding")]
SUMMARY = {"feature_set": FS, "app_target": APP_TARGET, "in_app": IN_APP,
           "flags_per_study_test": float(FL["test"].sum(1).mean()),
           "normal_studies_with_any_flag": float(FL["test"][normal].any(1).mean()) if normal.any() else float("nan"),
           "abnormal_studies_with_any_flag": float(FL["test"][yt[:, JA].any(1)].any(1).mean()) if yt[:, JA].any() else float("nan")}
for name, J in (("five common", J5), ("app labels", JA)):
    if not J:
        continue
    jj = [j for j in J if j in JA]
    flag = np.zeros_like(yt[:, J]); flag[:, [J.index(j) for j in jj]] = SC["test"][:, jj] >= THRS[APP_TARGET][jj]
    txt = TXT["test"][:, J]; g = yt[:, J]
    for k, v in (("text", txt), ("flags", flag), ("text OR flags", txt | flag)):
        tp = int((v & g).sum()); lo, hi = wilson(tp, int(g.sum()))
        SUMMARY[f"pooled_sens_{name}_{k}"] = [tp / max(g.sum(), 1), lo, hi]
        SUMMARY[f"pooled_spec_{name}_{k}"] = float((~v & ~g).sum() / max((~g).sum(), 1))
print("\nTEST system level:")
for k, v in SUMMARY.items():
    print(f"  {k}: " + (f"{v[0]:.3f} [{v[1]:.3f}, {v[2]:.3f}]" if isinstance(v, list) and len(v) == 3 and isinstance(v[0], float) else str(v)))

In [ ]:
# ---- export for the app: standardisation + weights + thresholds (JSON, plain numbers) and a torch head (for the ONNX step later)
J_ALL = [j for j, c in enumerate(LABELS) if P["per"][c]]
W = np.zeros((14, featset("val", FS).shape[1]), np.float32); B = np.zeros(14, np.float32)
for j in J_ALL:
    m = P["per"][LABELS[j]]["model"]; W[j], B[j] = m.coef_[0], m.intercept_[0]
mu, sd = P["scaler"].mean_.astype(np.float32), P["scaler"].scale_.astype(np.float32)
# check: the exported linear map reproduces the sklearn scores
Xt = featset("test", FS)
assert np.allclose(((Xt - mu) / sd) @ W[J_ALL].T + B[J_ALL], SC["test"][:, J_ALL], atol=1e-3), "exported probe does not reproduce the scores"
spec = {"version": NOTEBOOK_VERSION, "encoder_run": DEPLOYED, "checkpoint_step": ck.get("best_epoch"), "feature_set": FS,
        "feature_spec": "concat(encode()['cls'] [192], encode()['mean'] [192]" + (", encode()['grid'] [16x192] row-major flattened" if FS == "grid" else "") + ")",
        "preprocessing": f"uint8 512x512 -> /255 -> bilinear antialias resize to {IMG} -> 3 channels -> (x - 0.5) / 0.5",
        "labels": LABELS, "available": [bool(P["per"][c]) for c in LABELS], "in_app": IN_APP, "not_assessed": [c for c in PATH12 if c not in IN_APP],
        "score": "logit = ((feature - mean) / std) . W[label] + b[label]; flag if logit >= threshold[target][label]",
        "mean": mu.tolist(), "std": sd.tolist(), "W": W.tolist(), "b": B.tolist(),
        "threshold": {f"{t:.2f}": [None if np.isnan(v) else float(v) for v in THRS[t]] for t in TARGETS}, "app_target": f"{APP_TARGET:.2f}",
        "val_auc": [P["per"][c]["val_auc"] if P["per"][c] else None for c in LABELS], "test_summary": SUMMARY,
        "display_text": {c: f"possible {c.lower()}: check" for c in IN_APP},
        "limits": "Draft-report assistant, not a diagnostic device. Flags are tuned for ~90% sensitivity on validation and raise many false alarms; "
                  "labels listed as not assessed are never flagged."}
json.dump(spec, open(f"{WORK}/probe_{DEPLOYED}.json", "w"))

class ProbeHead(torch.nn.Module):
    """features [B, D] -> (logits [B, 14], flags [B, 14]) at the app thresholds; unavailable labels never flag."""
    def __init__(self, mu, sd, W, B, thr):
        super().__init__()
        for k, v in dict(mu=mu, sd=sd, W=W, B=B, thr=thr).items():
            self.register_buffer(k, torch.as_tensor(v, dtype=torch.float32))
    def forward(self, f):
        z = ((f - self.mu) / self.sd) @ self.W.T + self.B
        return z, z >= self.thr
thr_app = np.where(np.isin(np.arange(14), [LABELS.index(c) for c in IN_APP]), np.nan_to_num(THRS[APP_TARGET], nan=1e9), 1e9)
head = ProbeHead(mu, sd, W, B, thr_app)
torch.save({"state_dict": head.state_dict(), "labels": LABELS, "feature_set": FS, "spec_json": f"probe_{DEPLOYED}.json"}, f"{WORK}/probe_head_{DEPLOYED}.pt")
z, fl = head(torch.as_tensor(Xt, dtype=torch.float32))
assert (fl.numpy()[:, JA] == FL["test"]).all(), "torch head flags differ from the evaluated flags"
TABLE.to_csv(f"{WORK}/probe_test_table.csv")
json.dump({"summary": SUMMARY, "table": TABLE.reset_index().to_dict(orient="records"),
           "fits": {fs: FITS[fs]["mean_val_auc"] for fs in FITS}}, open(f"{WORK}/probe_results.json", "w"), indent=1, default=float)

import matplotlib; matplotlib.use("agg"); import matplotlib.pyplot as plt
lab = [c for c in LABELS if P["per"][c]]
fig, ax = plt.subplots(figsize=(10, 0.42 * len(lab) + 1.5))
yy = np.arange(len(lab))
ts = [float(TABLE.loc[c, f"sens@{APP_TARGET:.2f}"].split()[0]) for c in lab]
tx = [TABLE.loc[c, "text_sens"] for c in lab]
ax.barh(yy - 0.2, tx, 0.4, label="generated report alone", color="#7f8c8d")
ax.barh(yy + 0.2, ts, 0.4, label=f"probe flag (VAL target {APP_TARGET:.0%})", color="#c0392b")
ax.axvline(APP_TARGET, color="k", ls="--", lw=1)
ax.set_yticks(yy); ax.set_yticklabels([c + ("" if c in IN_APP else " (not in app)") for c in lab]); ax.invert_yaxis()
ax.set_xlabel("TEST sensitivity"); ax.legend(loc="lower right"); ax.grid(alpha=.3, axis="x")
ax.set_title(f"{DEPLOYED}: sensitivity of the report text vs the encoder probe flags")
plt.tight_layout(); plt.savefig(f"{WORK}/probe_sensitivity.png", dpi=130)
print(f"saved probe_{DEPLOYED}.json, probe_head_{DEPLOYED}.pt, probe_test_table.csv, probe_results.json, probe_sensitivity.png, {os.path.basename(FEAT_FILE)} "
      f"| total {(time.time() - T0) / 60:.1f} min\nPROBE DONE")